This notebook is the second of 2 notebooks that was used to test the eBEER model code against the example systems provided in Engel et al. 2020 (https://ui.adsabs.harvard.edu/abs/2020MNRAS.497.4884E/abstract).

This notebook does so by using the system parameters given in the Engel et al. 2020 to generate light curves with PHOEBE. It must be noted that PHOEBE 2.2+ does NOT support Beaming and so this notebook will not be testing the beaming effect.

Alternatively the first notebook tests against these systems using their Kepler data. For more details refer to the first notebook, "PaperSystemsTesting1.ipynb"

We start by importing the necessary packages:
 - numpy, pandas, matplotlib
 - phoebe
 - astropy.units - used to ensure PHOEBE understand what units we are working with
 - scipy.optimize.curve_fit - for fitting
 - testing_utils - a .py file with some general tools for constructing and printing PHOEBE binaries
 - phoebe_to_eBEER - .py file containing model code to take a phoebe binary and return eBEER model light curve

In [ ]:
import phoebe

import sys
sys.path.insert(1, '../scripts')
import phoebe_to_eBEER
import testing_utils as utils

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from astropy import units as u

from scipy.optimize import curve_fit

# System Parameters

We store all the parameters for the example sysems in the paper. They independently fit for these parameters with the eBEER model and PHOEBE. We store the parameters as a list of dataframes, where each dataframe refers to one system and contains 2 columns, the eBEER and PHOEBE fit parameters. This notebook will only work with the second column.

Metallicity, Linear Limb Darkening Coefficients, and Gravity Darkening Coefficients were found using MAST data and ExoCTK calculations with Kepler as the passband and the Phoeix ACES atmosphere model (unless otherwise noted)

In [ ]:
systems = ('KIC6292925', 'KIC5200778', 'KIC4931390')

KIC6292925 = {
    "Fitted Model": ['eBEER', 'PHOEBE'],
    "Porb": [13.6126, 13.6122],
    "t0_perpass": [874.04, 875.977],
    "M1": [1.9, 1.89],
    "M2": [0.50, 0.50],
    "R1": [1.8, 1.80],
    "R2": [1.3, 1.10],
    "T1": [8300, 8400],
    "T2": [6000, 7000],
    "incl": [71, 72],
    "per0": [157, 153],
    "ecc": [0.232, 0.2491],
    "metallicity": [0.087, 0.087],
    # Had to use Kurucz ATLAS9 Model as the Phoenix ACES model doesnt cover
    # Teff range
    "u1": [0.622, 0.606],
    "u2": [0.741, 0.616],
}

KIC5200778 = {
    "Fitted Model": ['eBEER', 'PHOEBE'],
    "Porb": [16.357, 16.3536],
    "t0_perpass": [806.6, 804.2],
    "M1": [1.2, 1.20],
    "M2": [0.40, 0.37],
    "R1": [2.4, 2.30],
    "R2": [0.14, 0.47],
    "T1": [6000, 6500],
    "T2": [4000, 3500],
    "incl": [23, 24],
    "per0": [-1, 2],
    "ecc": [0.35, 0.26],
    "metallicity": [None,None], # Not available in MAST table so defaulting to 0
    "u1": [0.741, 0.687],
    "u2": [0.577, 0.774],
}

KIC4931390 = {
    "Fitted Model": ['eBEER', 'PHOEBE'],
    "Porb": [7.6064, 7.6080],
    "t0_perpass": [798.44, 794.78],
    "M1": [1.20, 1.10],
    "M2": [0.10, 0.13],
    "R1": [1.48, 1.42],
    "R2": [0.7, 0.66],
    "T1": [6500, 6620],
    "T2": [5500, 6500],
    "incl": [50, 35],
    "per0": [158, 160],
    "ecc": [0.08, 0.040],
    "metallicity": [-0.313, -0.313],
    "u1": [0.695, 0.693],
    "u2": [0.736, 0.695],
}

dfs = (KIC6292925, KIC5200778, KIC4931390)

Then we define a function to construct the phoebe binary for one of the above systems.

In [ ]:
def grab_fit_params(df, idx):
    cur_fit_params = df.loc[idx]
    fit_type = cur_fit_params['Fitted Model']
    Porb = cur_fit_params['Porb']
    t0_perpass = cur_fit_params['t0_perpass']
    M1 = cur_fit_params['M1']
    M2 = cur_fit_params['M2']
    R1 = cur_fit_params['R1']
    R2 = cur_fit_params['R2']
    T1 = cur_fit_params['T1']
    T2 = cur_fit_params['T2']
    incl = cur_fit_params['incl']
    per0 = cur_fit_params['per0']
    ecc = cur_fit_params['ecc']
    u1 = cur_fit_params['u1']
    u2 = cur_fit_params['u2']
    abun = cur_fit_params['metallicity']
    pb = utils.get_phoebe_binary(Porb, t0_perpass, M1, M2, R1, R2, T1, T2, incl, per0, ecc, u1, u2, metallicity= abun)
    return fit_type, pb

# Select System
Here we select which system we want to work with and then construct all necessary objects for testing.

First we select a system by modifying $i$: (0 = KIC6292925, 1 = KIC5200778, 2 = KIC4931390). Then the following cell will construct a phoebe_binary_ for this system as well as a eBEERParams object which we use to quickly grab parameters like the Porb of the system as necessary. For more details on the eBEERParams object see my code in "..\scripts\phoebe_to_eBEER.py".

We want fit_type_ to be PHOEBE for this notebook.

In [ ]:
i = 1
target_name = systems[i]
df = pd.DataFrame(dfs[i])
fit_type_P, phoebe_binary_P = grab_fit_params(df, 1)

# eBEERParams object to grab frequently used values like Porb and t0
Pp = phoebe_to_eBEER.eBEERParams(phoebe_binary_P, times= 0) 

print(target_name, fit_type_P)
utils.print_pb_params(phoebe_binary_P)

# Setup PHOEBE Binary and Run LC Dataset

PHOEBE returns absolute flux so we must convert this to relative flux with the flux of the system with all effects turned off as the reference (this is the modulating eBEER calculates).

To do this we initialize the first phoebe_binary and simulate the lightcurve with the effects on.

We set passband to Kepler and use Phoenix as the atmosphere model. We can also conveniently turn of eclipses with `'eclipse_method'='only_horizon'`.

In [ ]:
time_P = np.linspace(Pp.t0, Pp.t0 + 2*Pp.Porb,201)

phoebe_binary_P.add_dataset('lc', times= time_P, dataset='lc01', overwrite=True)
phoebe_binary_P.set_value_all('ntriangles', 5000)
phoebe_binary_P['eclipse_method'] = 'only_horizon'
phoebe_binary_P['passband'] = 'Kepler:mean'
phoebe_binary_P.set_value_all('atm', 'phoenix')
phoebe_binary_P.run_compute()

We now initialize the second binary and simulate the light curve with all effects off.

We can turn off reflection with `'irrad_method'='none'` and ellipsoidal with `'distortion_method'='sphere'`. We have to turn off ellipsoidal for both stars which can be easily accomplished with `set_value_all()`

In [ ]:
_, phoebe_binary_P2 = grab_fit_params(df, 1)
phoebe_binary_P2.add_dataset('lc', times=0, label= 'lc01')
phoebe_binary_P2.set_value_all('ntriangles', 5000)
phoebe_binary_P2['eclipse_method'] = 'only_horizon'
phoebe_binary_P2['passband'] = 'Kepler:mean'
phoebe_binary_P2.set_value_all('atm', 'phoenix')
phoebe_binary_P2.set_value('irrad_method', 'none')
phoebe_binary_P2.set_value_all('distortion_method', value='sphere')
phoebe_binary_P2.run_compute()

We then compute relative flux and conver to ppm

In [ ]:
flux = phoebe_binary_P[f'fluxes@lc01@latest@model'].value
ref_flux = phoebe_binary_P2['fluxes@lc01@latest@model'].value
relative_flux_ = (flux - ref_flux)/ref_flux * 1e6

In [ ]:
plt.plot(time_P, relative_flux_)

# Run Fit

The next cell contains fit_eBEER_PHOEBE which we will pass to scipy.curve_fit we have the fitting code. We scale all values to range (0,1) using fit_scaling_Kepler to improve fitting time and accuracy. Faigler & Mazeh 2011 (https://ui.adsabs.harvard.edu/abs/2011MNRAS.415.3921F/abstract) offers loose suggested ranges for the scaling coefficients. I loosen the ranges a little more as the suggested values are not hard limits, and I found this to still give satisfactory results.

In [ ]:
def fit_scaling_PHOEBE(time_shift, arefl1, arefl2):
    scaled = np.zeros(3)
    scaled[0] = time_shift * Pp.Porb
    scaled[1] = arefl1 * 7
    scaled[2] = arefl2 * 7
    return scaled

def fit_eBEER_PHOEBE(t, time_shift, arefl1, arefl2):
   
  scaled = fit_scaling_PHOEBE(time_shift, arefl1, arefl2)

  flux = phoebe_to_eBEER.get_eBEER_lc(phoebe_binary_P, (t + scaled[0]) % Pp.Porb, 0, 0, scaled[1], scaled[2])
  
  return flux

Now we run the below cell to help us make a good initial guess for the time_shift. The resulting plot has beaming and reflection OFF.

In [ ]:
flux = phoebe_to_eBEER.get_eBEER_lc(phoebe_binary_P, time_P, 0,0,0,0)
phase = (time_P) % Pp.Porb

plt.figure(figsize=(16,5), label= 'Gauge initial guess for time shift')
plt.subplot(121)
plt.plot(time_P, relative_flux_, 'b.', markersize= 1, label= 'Flux to fit')
plt.plot(time_P, flux, 'r', label= 'eBEER w/o beaming or reflection')
plt.legend()

plt.subplot(122)
plt.plot(phase, relative_flux_, 'b.', markersize= 1, label= 'Flux to fit')
plt.plot(phase, flux, 'r', label= 'eBEER w/o beaming or reflection')
plt.legend()

Now we can run the fit and view the results

In [ ]:
popt_P, pcov_P = curve_fit(fit_eBEER_PHOEBE, time_P, relative_flux_, 
                           p0= ((0)/Pp.Porb, 0.5, 0.5), 
                           bounds= ([0, 0, 0], [1, 1, 1]))

In [ ]:
fit_params_P = ('Time Shift', 'Alpha Reflective 1', 'Alpha Reflective 2')
rescaled_P = fit_scaling_PHOEBE(popt_P[0], popt_P[1], popt_P[2])
for name,value in zip(fit_params_P, rescaled_P):
    print(name+':', value)

In [ ]:
fit_flux_P = fit_eBEER_PHOEBE(time_P, popt_P[0], popt_P[1], popt_P[2])

plt.figure(figsize=(8, 5))
plt.plot(time_P, relative_flux_, 'b.', markersize=1, label= 'PHOEBE LC')
plt.plot(time_P, fit_flux_P, 'r', markersize=1, label= 'PHOEBE Fit')
plt.title(f'eBEER Fit for PHOEBE LC: {target_name}')
plt.xlabel('Time (days)')
plt.ylabel('Flux')
plt.legend()

In [ ]:
dif = np.abs(fit_flux_P - relative_flux_)
MSD = np.sqrt(np.mean(dif**2))

print(f'Mean Diff: {np.mean(dif)} ppm   STD: {np.std(dif)} ppm')
print(f'MSD: {MSD} ppm')